# 一、加载模型和预处理器

In [22]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
Qwen2_5_VL_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype="auto", device_map="auto"
)

processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
Qwen2_VL_model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct", torch_dtype="auto", device_map="auto"
)

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
path = "OpenGVLab/InternVL2_5-1B"
internvl_model = AutoModel.from_pretrained(
    path,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="cuda:0"
    ).eval()

# 二、准备输入数据


In [24]:
image_path = "./example.png"
prompt = "Describe this image."


## 2.1 准备Qwen2VL和Qwen2.5VL的输入数据

In [25]:
from qwen_vl_utils import process_vision_info

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image_path,
            },
            {"type": "text", "text": prompt},
        ],
    }
]

In [26]:
# 创建模型输入
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)

# 确保所有输入都在正确的设备上
device = next(model.parameters()).device
inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

In [27]:
# 修改torchinfo的输入格式
Qwen_model_inputs_for_summary = {
    'input_ids': inputs['input_ids'],
    'attention_mask': inputs['attention_mask'],
    'pixel_values': inputs['pixel_values'],
    'image_grid_thw': inputs['image_grid_thw'],
}


# 创建一个元组作为模型的输入
Qwen_model_inputs_for_thop = (
    inputs['input_ids'],          # input_ids
    inputs['attention_mask'],     # attention_mask
    None,                         # position_ids
    None,                         # past_key_values
    None,                         # inputs_embeds
    None,                         # labels
    None,                         # use_cache
    None,                         # output_attentions
    None,                         # output_hidden_states
    None,                         # return_dict
    inputs['pixel_values'],       # pixel_values
    None,                         # pixel_values_videos
    inputs['image_grid_thw'],     # image_grid_thw
)

## 2.2 准备InternVL的输入数据

In [4]:
from thop import profile
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoTokenizer

from internvl.conversation import get_conv_template

tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True, use_fast=False, device_map="cuda:0")

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    # calculate the existing image aspect ratio
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    # find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    # calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    # resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        # split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

In [5]:
device = next(internvl_model.parameters()).device  # 获取模型所在的设备

# 1. 准备图像输入
pixel_values = load_image(image_path, max_num=12).to(torch.bfloat16).to(device)

# 2. 准备文本输入
question = '<image>\n'+prompt
template = get_conv_template(internvl_model.template)
template.system_message = internvl_model.system_message
template.append_message(template.roles[0], question)
template.append_message(template.roles[1], None)
query = template.get_prompt()

# 3. 处理图像标记
IMG_START_TOKEN = '<img>'
IMG_END_TOKEN = '</img>'
IMG_CONTEXT_TOKEN = '<IMG_CONTEXT>'
num_patches = pixel_values.shape[0]  # 13
tokens_per_patch = internvl_model.num_image_token  # 256
image_tokens = IMG_START_TOKEN + IMG_CONTEXT_TOKEN * (tokens_per_patch * num_patches) + IMG_END_TOKEN
query = query.replace('<image>', image_tokens, 1)

# 4. 准备模型输入
model_inputs = tokenizer(query, return_tensors='pt')
input_ids = model_inputs['input_ids'].to(device)
attention_mask = model_inputs['attention_mask'].to(device)

# 5. 创建image_flags
img_context_token_id = tokenizer.convert_tokens_to_ids(IMG_CONTEXT_TOKEN)
internvl_model.img_context_token_id = img_context_token_id

# 修改 image_flags 的处理方式
image_flags = (input_ids == img_context_token_id).to(device)
# 重塑 image_flags 以匹配 vit_embeds 的维度
reshaped_image_flags = torch.zeros(num_patches, 256, dtype=torch.bool, device=device)
reshaped_image_flags[:, :] = True  # 所有patch的所有位置都设为True

# 调试信息
# print(f"pixel_values shape: {pixel_values.shape}")  # [13, 3, 448, 448]
# print(f"vit_embeds shape: {internvl_model.extract_feature(pixel_values).shape}")  # [13, 256, 896]
# print(f"input_ids shape: {input_ids.shape}")  # [1, sequence_length]
# print(f"reshaped_image_flags shape: {reshaped_image_flags.shape}")  # [13, 256]

In [11]:
InternVL_model_inputs_for_summary = {
    'pixel_values': pixel_values,
    'input_ids': input_ids,
    'attention_mask': attention_mask,
    'image_flags': reshaped_image_flags,
}

# 6. 创建完整的模型输入
InternVL_model_inputs_for_thop = (
    pixel_values,        # pixel_values
    input_ids,          # input_ids
    attention_mask,     # attention_mask
    None,               # position_ids
    reshaped_image_flags,  # 使用重塑后的 image_flags
    None,               # past_key_values
    None,               # labels
    None,               # use_cache
    None,               # output_attentions
    None,               # output_hidden_states
    None,               # return_dict
)

# 计算数据


In [28]:
# model = internvl_model
# model_inputs_for_summary = InternVL_model_inputs_for_summary
# model_inputs_for_thop = InternVL_model_inputs_for_thop

model = Qwen2_5_VL_model
# model = Qwen2_VL_model
model_inputs_for_summary = Qwen_model_inputs_for_summary
model_inputs_for_thop = Qwen_model_inputs_for_thop


## 使用torchinfo展开模型结构和计算参数量

In [29]:
from torchinfo import summary
# 调用torchinfo
summary(
    model, 
    input_data=model_inputs_for_summary,
    col_names=["input_size", "output_size", "num_params", "params_percent", "kernel_size", "mult_adds"],
    col_width=20,
    depth=10,
    row_settings=["ascii_only", "var_names"]
)

Layer (type (var_name))                                                     Input Shape          Output Shape         Param #              Param %              Kernel Shape         Mult-Adds
Qwen2_5_VLForConditionalGeneration (Qwen2_5_VLForConditionalGeneration)     --                   [1, 1]               --                   -14.22%              --                   --
+ Qwen2_5_VLModel (model)                                                   --                   --                   (recursive)          (recursive)          --                   --
|    + Embedding (embed_tokens)                                             [1, 761]             [1, 761, 2048]       311,164,928            7.11%              --                   311,164,928
+ Qwen2_5_VisionTransformerPretrainedModel (visual)                         [2944, 1176]         [736, 2048]          --                        --              --                   --
|    + Qwen2_5_VisionPatchEmbed (patch_embed)                   

## 使用thop计算FLOPs和参数量

In [30]:
from thop import profile

# 计算FLOPs
Flops, params = profile(model, inputs=model_inputs_for_thop)

print('FLOPs: %.4fG' % (Flops / 1e9))  # 计算量
print('Params: %.4fM' % (params / 1e6)) # 参数量

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv3d'>.
[INFO] Register count_linear() for <class 'torch.nn.modules.linear.Linear'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.
FLOPs: 4234.2747G
Params: 3754.3903M
